In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Consistency Evaluation for Leela Eval Project

This notebook evaluates the consistency of the research project at `/net/scratch2/smallyan/leela_eval`.

## Evaluation Criteria:
- **CS1**: Conclusions vs Original Results - Do documented conclusions match implementation results?
- **CS2**: Implementation Follows Plan - Does the implementation follow all plan steps?

In [2]:
# Check CUDA availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    device = 'cuda'
else:
    device = 'cpu'
print(f"Using device: {device}")

CUDA available: True
CUDA device: NVIDIA H200 NVL
Using device: cuda


In [3]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/leela_eval'
for root, dirs, files in os.walk(repo_path):
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:20]:  # Limit files shown per directory
        print(f'{subindent}{file}')
    if len(files) > 20:
        print(f'{subindent}... and {len(files) - 20} more files')

leela_eval/
  lc0.onnx
  plan.md
  documentation.pdf
  .gitmodules
  pyproject.toml
  lc0-original.onnx
  768x15x24h-t82-swa-7464000.pb
  .gitignore
  CodeWalkthrough.md
  768x15x24h-t82-swa-7464000.pb.gz
  iteration_model/
    interesting_puzzles.pkl
    lc0.onnx
    lc0-random.onnx
    LD2.onnx
    unfiltered_puzzles.pkl
    lc0-original.onnx
  lc0_bin/
    lc0.tar.gz
  src/
    leela_logit_lens/
      __init__.py
      tournament/
        logit_lens_engine.py
        constants.py
        __pycache__/
          logit_lens_engine.cpython-312.pyc
          constants.cpython-311.pyc
          logit_lens_engine.cpython-311.pyc
      tools/
        evaluate_puzzles.py
        plotting_helpers.py
        utils.py
        sample_positions.py
        evaluate_concepts.py
        puzzle_history_augmentation.py
        concept_spec.py
        tournament.py
        __pycache__/
          evaluate_concepts.cpython-311.pyc
          tournament.cpython-311.pyc
          evaluate_puzzles.cpython-31

## Step 1: Read the Plan File

Let's first read the plan.md file to understand what was planned for implementation.

In [4]:
# Read the plan file
plan_path = os.path.join(repo_path, 'plan.md')
with open(plan_path, 'r') as f:
    plan_content = f.read()
print(plan_content)

# Plan
## Objective
Investigate how neural networks progressively build understanding across layers by extending the logit lens technique to analyze the policy network of Leela Chess Zero, examining whether representations are refined through smooth gradual processes or more complex computational mechanisms involving iterative inference with distinct phases.

## Hypothesis
1. Neural networks perform iterative inference with capability progression occurring in distinct computational phases rather than smooth gradual refinement
2. Leela's inference process combines algorithmic computation with learned heuristic priors, where safety-oriented heuristics can override tactical solutions

## Methodology
1. Extend logit lens to Post-LN transformer architectures by applying zero ablation to sublayer outputs beyond layer ℓ while preserving subsequent layer normalizations and ablating layer normalization biases
2. Analyze T82-768x15x24h transformer model with 15 layers and 768-dimensional embeddi

## Step 2: Read the Documentation File

Now let's read the documentation.pdf to understand the conclusions drawn from the project.